# Libraries

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns


from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

In [ ]:
# dispongo la carpeta con funciones e importo ternara. 
from pathlib import Path
import sys
import importlib


# Obtiene la ruta del directorio actual y sube un nivel (.parent)
raiz_proyecto = Path().resolve().parent.parent


# Agrega la ruta al sistema si no está ya incluida
if str(raiz_proyecto) not in sys.path:
  sys.path.append(str(raiz_proyecto))

# Importa tu librería o módulo
from funciones.ternaria import ternaria
from funciones.graficos_clusters_pdf  import generar_pdf_clusters
from funciones.distribucion_clusters_pdf  import generar_pdf_distribuciones_clusters

# URLs y Constantes

In [3]:
BASE_URL = os.path.join('/mnt/', 'c')
BASE_DATA_URL = os.path.join('/mnt/', 'e')
COMP_URL = os.path.join(BASE_URL, 'Users', 'marco', 'Desktop', 'MASTER', 'MASTER', 'DMEyF', 'COMP1')
DATA_FOLDER = os.path.join(BASE_DATA_URL, 'DATASETS', 'DMEyF')
DATA_URL = os.path.join(DATA_FOLDER, 'competencia_01_crudo.csv')
DATA_TERNARIA_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria.csv')
DATA_DICT_URL= os.path.join(DATA_FOLDER, 'data_dict.csv')

SEED = 230047

In [4]:
os.listdir(DATA_FOLDER)

['competencia_01_crudo.csv',
 'competencia_01_ternaria.csv',
 'data_dict.csv',
 'DiccionarioDatos_2026.ods']

# UTILS

In [5]:
def van_dongen_normalized(contingency):
    n = contingency.values.sum()
    sum_max_rows = contingency.max(axis=1).sum()  # para cada label, max sobre clusters
    sum_max_cols = contingency.max(axis=0).sum()  # para cada cluster, max sobre labels
    max_row_total = contingency.sum(axis=1).max()
    max_col_total = contingency.sum(axis=0).max()
    
    vdn = (2*n - sum_max_rows - sum_max_cols) / (2*n - max_row_total - max_col_total)
    return vdn

In [6]:
def consolidar_columnas(
    df,
    columnas_base,
    diccionario_consolidaciones
):

    df_salida = df[columnas_base].copy()

    for columna_nueva, configuracion in diccionario_consolidaciones.items():

        columnas_origen = configuracion["columnas"]
        operacion = configuracion["operacion"].lower()

        if operacion == "sum":
            df_salida[columna_nueva] = df[columnas_origen].sum(
                axis=1,
                min_count=1
            )

        elif operacion == "avg":
            df_salida[columna_nueva] = df[columnas_origen].mean(axis=1)

        elif operacion == "max":
            df_salida[columna_nueva] = df[columnas_origen].max(axis=1)

        elif operacion == "min":
            df_salida[columna_nueva] = df[columnas_origen].min(axis=1)

        else:
            raise ValueError(
                f"Operación no válida para {columna_nueva}: {operacion}"
            )

    df_salida["ternaria"] = df["ternaria"]

    return df_salida

# LEEMOS DATOS

In [7]:
df = pd.read_csv(
    DATA_TERNARIA_URL,
    dtype={'numero_de_cliente': 'int32', 'foto_mes': 'int32'}
)

/tmp/ipykernel_137183/3175415450.py:1: DtypeWarning: Columns (0: ternaria) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


# ANALISIS

## CONSOLIDAMOS INFO

In [8]:
def agregar_mes_baja(
    df,
    cliente_col="numero_de_cliente",
    mes_col="foto_mes"
):
    resultado = df.copy()

    ultimo_mes_dataset = resultado[mes_col].max()

    ultimo_mes_cliente = (
        resultado
        .groupby(cliente_col)[mes_col]
        .max()
    )

    mes_baja = pd.Series(
        pd.NA,
        index=ultimo_mes_cliente.index,
        dtype="Int64"
    )

    clientes_baja = ultimo_mes_cliente < ultimo_mes_dataset
    ultimo_mes = ultimo_mes_cliente.loc[clientes_baja]

    # Calcula correctamente el mes siguiente, incluyendo diciembre → enero
    mes_baja.loc[clientes_baja] = (
        (ultimo_mes // 100 + (ultimo_mes % 100 == 12)) * 100
        + (ultimo_mes % 100) % 12 + 1
    ).astype("Int64")

    resultado["mes_baja"] = (
        resultado[cliente_col]
        .map(mes_baja)
        .astype("Int64")
    )

    return resultado

In [9]:
# Agregamos columna con mes q se da de baja quien asi lo haya hecho.
df = agregar_mes_baja(df)

In [10]:
diccionario_consolidaciones = {
    "msueldo_acreditado": {
        "columnas": [
            "mpayroll",
            "mpayroll2"
        ],
        "operacion": "sum"
    },

    "mconsumo_tarjetas_credito": {
        "columnas": [
            "mtarjeta_visa_consumo",
            "mtarjeta_master_consumo"
        ],
        "operacion": "sum"
    },

    "mdeuda_tarjetas_credito": {
        "columnas": [
            "Visa_msaldototal",
            "Master_msaldototal"
        ],
        "operacion": "sum"
    },

    "mtransferencias_total": {
        "columnas": [
            "mtransferencias_recibidas",
            "mtransferencias_emitidas"
        ],
        "operacion": "sum"
    },

    "mdebitos_automaticos": {
        "columnas": [
            "mcuenta_debitos_automaticos",
            "mttarjeta_visa_debitos_automaticos",
            "mttarjeta_master_debitos_automaticos"
        ],
        "operacion": "sum"
    },

    "mlimite_compra_tarjetas_credito": {
        "columnas": [
            "Visa_mlimitecompra",
            "Master_mlimitecompra"
        ],
        "operacion": "sum"
    },

    "mlimite_financiacion_tarjetas_credito": {
        "columnas": [
            "Visa_mfinanciacion_limite",
            "Master_mfinanciacion_limite"
        ],
        "operacion": "sum"
    },

    "ctarjetas_en_mora": {
        "columnas": [
            "Visa_delinquency",
            "Master_delinquency"
        ],
        "operacion": "sum"
    },

    "mpagos_tarjetas_credito": {
        "columnas": [
            "Visa_mpagado",
            "Master_mpagado"
        ],
        "operacion": "sum"
    },

    "madelantos_efectivo_tarjetas_credito": {
        "columnas": [
            "Visa_madelantopesos",
            "Visa_madelantodolares",
            "Master_madelantopesos",
            "Master_madelantodolares"
        ],
        "operacion": "sum"
    },

    "mplazo_fijo_total": {
        "columnas": [
            "mplazo_fijo_pesos",
            "mplazo_fijo_dolares"
        ],
        "operacion": "sum"
    },

    "minversion1_total": {
        "columnas": [
            "minversion1_pesos",
            "minversion1_dolares"
        ],
        "operacion": "sum"
    },

    "mdescuentos_total": {
        "columnas": [
            "mcajeros_propios_descuentos",
            "mtarjeta_visa_descuentos",
            "mtarjeta_master_descuentos"
        ],
        "operacion": "sum"
    },
    "mpagodeservicios_consolidado": {
        "columnas": [
            "mpagodeservicios",
            "mpagomiscuentas",
        ],
        "operacion": "sum"
    },
    "cpagodeservicios_consolidado": {
        "columnas": [
            "cpagodeservicios",
            "cpagomiscuentas",
        ],
        "operacion": "sum"
    },

}
# columnas_base = [
#         "numero_de_cliente",
#         "foto_mes",
#         "mes_baja",
#         "active_quarter",
#         "cliente_vip",

#         # Actividad general y digital
#         "ctrx_quarter",
#         "chomebanking_transacciones",
#         "cmobile_app_trx",

#         # Transferencias
#         "mtransferencias_recibidas",
#         "mtransferencias_emitidas",

#         # Pagos habituales
#         "mpagodeservicios",
#         "mpagomiscuentas",
#         "mcuenta_debitos_automaticos",

#         # Extracciones
#         "mextraccion_autoservicio",

#         # Mantenimiento, saldo y productos
#         "mcomisiones_mantenimiento",
#         "mcuentas_saldo",
#         "cproductos"
#     ]

In [26]:
columnas_base = [
    "numero_de_cliente",
    "foto_mes",
    "mes_baja",
    "active_quarter",
    "cliente_vip",
    "internet",
    "cliente_edad",
    "cliente_antiguedad",

    # Rentabilidad y comisiones ya consolidadas
    "mrentabilidad",
    "mrentabilidad_annual",
    "mcomisiones",

    # Productos y cuentas
    "cproductos",
    "tcuentas",
    "ccuenta_corriente",
    "ccaja_ahorro",
    "mcuentas_saldo",

    # Tarjeta de débito
    "ctarjeta_debito_transacciones",
    "mautoservicio",

    # Tarjetas de crédito
    "ctarjeta_visa_transacciones",
    "ctarjeta_master_transacciones",

    # Préstamos: cada monto ya es el total de su categoría
    "mprestamos_personales",


    # Inversiones
    "minversion2",

    # Cantidades asociadas a sueldo
    "cpayroll_trx",
    "cpayroll2_trx",

    # Cantidades de débitos automáticos
    "ccuenta_debitos_automaticos",
    "ctarjeta_visa_debitos_automaticos",
    "ctarjeta_master_debitos_automaticos",

    # Cantidades de descuentos
    "ccajeros_propios_descuentos",
    "ctarjeta_visa_descuentos",
    "ctarjeta_master_descuentos",

    # Transferencias
    "ctransferencias_recibidas",
    "ctransferencias_emitidas",

    # Extracciones: monto total ya existente
    "cextraccion_autoservicio",
    "mextraccion_autoservicio",


    # Canales
    "ccallcenter_transacciones",
    "chomebanking_transacciones",
    "catm_trx",
    "catm_trx_other",
    "ctrx_quarter",
    "cmobile_app_trx",

    # Mastercard
    "Master_status",
    "Master_cconsumos",
    "Master_cadelantosefectivo",
    "Master_mpagominimo",

    # Visa
    "Visa_status",
    "Visa_cconsumos",
    "Visa_cadelantosefectivo",
    "Visa_mpagominimo",

    "ternaria"
]

In [27]:
df_resumen = consolidar_columnas(df,columnas_base, diccionario_consolidaciones)

In [28]:
df_resumen.columns

Index(['numero_de_cliente', 'foto_mes', 'mes_baja', 'active_quarter',
       'cliente_vip', 'internet', 'cliente_edad', 'cliente_antiguedad',
       'mrentabilidad', 'mrentabilidad_annual', 'mcomisiones', 'cproductos',
       'tcuentas', 'ccuenta_corriente', 'ccaja_ahorro', 'mcuentas_saldo',
       'ctarjeta_debito_transacciones', 'mautoservicio',
       'ctarjeta_visa_transacciones', 'ctarjeta_master_transacciones',
       'mprestamos_personales', 'minversion2', 'cpayroll_trx', 'cpayroll2_trx',
       'ccuenta_debitos_automaticos', 'ctarjeta_visa_debitos_automaticos',
       'ctarjeta_master_debitos_automaticos', 'ccajeros_propios_descuentos',
       'ctarjeta_visa_descuentos', 'ctarjeta_master_descuentos',
       'ctransferencias_recibidas', 'ctransferencias_emitidas',
       'cextraccion_autoservicio', 'mextraccion_autoservicio',
       'ccallcenter_transacciones', 'chomebanking_transacciones', 'catm_trx',
       'catm_trx_other', 'ctrx_quarter', 'cmobile_app_trx', 'Master_status',


In [29]:
mask_baja = df_resumen['ternaria'].str.contains('BAJA') 

In [30]:
proporcion_ceros = (
    df_resumen.loc[mask_baja].select_dtypes(include="number")
      .eq(0)
      .sum()
      .div(len(df_resumen.loc[mask_baja]))
      .sort_values(ascending=False)
)
proporcion_ceros

cpayroll2_trx                      0.999673
cliente_vip                        0.999237
ccajeros_propios_descuentos        0.991494
minversion1_total                  0.990403
minversion2                        0.990185
                                     ...   
cliente_antiguedad                      0.0
cproductos                              0.0
tcuentas                                0.0
ccuenta_corriente                       0.0
mlimite_compra_tarjetas_credito         0.0
Length: 63, dtype: Float64

# k_MEANS

In [32]:
# columnas que no tienen valor informativo
columnas_excluir = [
    "numero_de_cliente",
    "foto_mes",
    "ternaria",
    "mes_baja"
]
# Siempre agrego n registros de no baja para ver si el clustering los agrupa separado de los demas. 
def get_random_continua(k, mask_baja):
    indices_random_continua = (
        df.loc[df["ternaria"] == "continua"]
        .sample(
            n=round(mask_baja.sum() / k),
            random_state=SEED
        )
        .index)
    mask_random_continua = df.index.isin(indices_random_continua)
    return mask_random_continua

In [35]:
# columnas que no tienen valor informativo

sparsity_rates = [0.2, 0.5, 0.8]
ks = [3,4,5,6]
resultados = []
# agrego registros q no son de baja, quiero ver si el clustering los distingue
mask_random = get_random_continua(5,mask_baja) #tomamos una proporcion siempre como si k fuera 5
df_baja = df_resumen.loc[mask_baja|mask_random].copy()
df_baja.fillna(0, inplace=True) #naively filll with 0s
labels_reales = df_baja['ternaria']
for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []  
    for columna in proporcion_ceros.items():
        if columna[1]>rate:
            sparse_columns.append(columna[0])
        else :
            non_sparse_columns.append(columna[0])

    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]

    X = df_baja.drop(columns = _columnas_excluir).select_dtypes(include='number')
    X_escalado = StandardScaler().fit_transform(X)

    for k in ks:

        kmeans = KMeans(
        n_clusters=k+1,
        n_init=20,
        random_state=SEED
        )
        column_name = f'k-mean-{k}-spars-{rate*10:.0f}'
        df_baja[column_name] = kmeans.fit_predict(X_escalado)

        labels = kmeans.labels_
        resultados.append({
            'column_name':column_name,
            "sparsity_rate": rate,
            "k_baja": k,
            "clusters_totales": k + 1,
            "tamanios_clusters": pd.Series(labels).value_counts().sort_index().to_dict(),
            "ncolumnas_usadas": X.shape[1],
            "columnas_usadas": non_sparse_columns,
            "columnas_excluidas_sparse": len(sparse_columns),
            "inercia": kmeans.inertia_,                                                   # menor
            "iteraciones": kmeans.n_iter_,
            "silhouette": silhouette_score(X_escalado, labels),                          # mayor
            "davies_bouldin": davies_bouldin_score(X_escalado, labels),                   # menor
            "calinski_harabasz": calinski_harabasz_score(X_escalado, labels)              # mayor
        })
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados.sort_values(
    "silhouette",
    ascending=False
)

,column_name,sparsity_rate,k_baja,clusters_totales,tamanios_clusters,ncolumnas_usadas,columnas_usadas,columnas_excluidas_sparse,inercia,iteraciones,silhouette,davies_bouldin,calinski_harabasz
5,k-mean-4-spars-5,0.5,4,5,"{0: 1316, 1: 7662, 2: 1872, 3: 134, 4: 20}",24,"[chomebanking_transacciones, Visa_mpagominimo,...",40,173156.340932,8,0.364548,1.193011,1444.136127
4,k-mean-3-spars-5,0.5,3,4,"{0: 1316, 1: 7666, 2: 1888, 3: 134}",24,"[chomebanking_transacciones, Visa_mpagominimo,...",40,185115.233696,8,0.359826,1.339687,1564.410110
8,k-mean-3-spars-8,0.8,3,4,"{0: 7896, 1: 1316, 2: 4, 3: 1788}",45,"[mautoservicio, ctarjeta_debito_transacciones,...",23,357155.565910,7,0.307283,1.319707,1417.000433
0,k-mean-3-spars-2,0.2,3,4,"{0: 7704, 1: 1844, 2: 1445, 3: 11}",17,"[Master_cadelantosefectivo, active_quarter, Vi...",43,145861.490704,14,0.255823,1.588933,1035.853534
3,k-mean-6-spars-2,0.2,6,7,"{0: 1536, 1: 11, 2: 134, 3: 7181, 4: 8, 5: 848...",17,"[Master_cadelantosefectivo, active_quarter, Vi...",43,118267.771686,21,0.251392,1.257101,1066.219707
1,k-mean-4-spars-2,0.2,4,5,"{0: 1907, 1: 1317, 2: 7635, 3: 11, 4: 134}",17,"[Master_cadelantosefectivo, active_quarter, Vi...",43,135319.775445,18,0.229469,1.375448,1051.549622
7,k-mean-6-spars-5,0.5,6,7,"{0: 134, 1: 1315, 2: 11, 3: 2773, 4: 37, 5: 50...",24,"[chomebanking_transacciones, Visa_mpagominimo,...",40,153069.644598,14,0.220469,1.386621,1329.413194
6,k-mean-5-spars-5,0.5,5,6,"{0: 2784, 1: 5073, 2: 20, 3: 1677, 4: 1316, 5:...",24,"[chomebanking_transacciones, Visa_mpagominimo,...",40,163958.169731,8,0.214765,1.375207,1343.411393
11,k-mean-6-spars-8,0.8,6,7,"{0: 5061, 1: 2760, 2: 5, 3: 1713, 4: 1316, 5: ...",45,"[mautoservicio, ctarjeta_debito_transacciones,...",23,308010.416748,7,0.181802,1.430129,1113.763147
2,k-mean-5-spars-2,0.2,5,6,"{0: 1311, 1: 3266, 2: 5179, 3: 1103, 4: 134, 5...",17,"[Master_cadelantosefectivo, active_quarter, Vi...",43,127588.102543,12,0.180020,1.523042,1025.429840


In [41]:
pd.crosstab(
        df_baja["k-mean-4-spars-2"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
k-mean-4-spars-2,,,,
0,581,501,825,1907
1,756,546,15,1317
2,3693,2962,980,7635
3,3,3,5,11
4,70,55,9,134
All,5103,4067,1834,11004


In [42]:
columna_cluster = 'k-mean-4-spars-2'
file_name = f'graficos-{columna_cluster}-sobre-consolidado2-resumen.pdf'
output_file = os.path.join( os.getcwd(), 'graficos_clusters', file_name)
generar_pdf_clusters(
    df_variables=df_resumen,
    df_baja=df_baja,
    columna_cluster=columna_cluster,
    archivo_salida=output_file,
    banda="ic95",
    data_dict=DATA_DICT_URL,
    diccionario_consolidaciones=diccionario_consolidaciones)

PosixPath('/home/marco/code/DMEyF/dmeyf2026/mis_pruebas/graficos_clusters/graficos-k-mean-4-spars-2-sobre-consolidado2-resumen.pdf')

# Hierarchical


In [39]:
sparsity_rates = [0.2, 0.5, 0.8]
ks = [3,4,5,6]
linkages = ["ward", "complete", "average", "single"]
metrics = ["euclidean", "manhattan", "cosine"]

resultados_h = []

# agrego registros q no son de baja, quiero ver si el clustering los distingue
mask_random = get_random_continua(5,mask_baja) #tomamos una proporcion siempre como si k fuera 5
df_baja_h = df_resumen.loc[mask_baja|mask_random].copy()
df_baja_h.fillna(0, inplace=True) #naively filll with 0s
labels_reales = df_baja_h['ternaria']

for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []

    for columna in proporcion_ceros.items():
        if columna[1] > rate:
            sparse_columns.append(columna[0])
        else:
            non_sparse_columns.append(columna[0])

    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]

    for k in ks:
        X = (
            df_baja_h
            .drop(columns=_columnas_excluir)
            .select_dtypes(include="number")
        )

        X_escalado = StandardScaler().fit_transform(X)

        for linkage in linkages:
            for metric in metrics:

                # Ward solamente admite distancia euclídea
                if linkage == "ward" and metric != "euclidean":
                    continue

                modelo = AgglomerativeClustering(
                    n_clusters=k + 1,
                    linkage=linkage,
                    metric=metric
                )

                labels = modelo.fit_predict(X_escalado)

                column_name = (
                    f"agg-{k}-{linkage}-{metric}-spars-{rate*10:.0f}"
                )

                df_baja_h[column_name] = labels

                resultados_h.append({
                    "column_name": column_name,
                    "sparsity_rate": rate,
                    "k_baja": k,
                    "clusters_totales": k + 1,
                    "linkage": linkage,
                    "metric": metric,
                    "columnas_usadas": X.shape[1],
                    "columnas_usadas": non_sparse_columns,
                    "columnas_excluidas_sparse": len(sparse_columns),
                    "silhouette": silhouette_score(X_escalado, labels),
                    "davies_bouldin": davies_bouldin_score(
                        X_escalado,
                        labels
                    ),
                    "calinski_harabasz": calinski_harabasz_score(
                        X_escalado,
                        labels
                    ),
                    "tamanios_clusters": (
                        pd.Series(labels)
                        .value_counts()
                        .sort_index()
                        .to_dict()
                    )
                })

tabla_resultados_h = pd.DataFrame(resultados_h)

/tmp/ipykernel_137183/4199584670.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baja_h[column_name] = labels
/tmp/ipykernel_137183/4199584670.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baja_h[column_name] = labels
/tmp/ipykernel_137183/4199584670.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfra

In [40]:
tabla_resultados_h.sort_values(by='silhouette', ascending=False)

,column_name,sparsity_rate,k_baja,clusters_totales,linkage,metric,columnas_usadas,columnas_excluidas_sparse,silhouette,davies_bouldin,calinski_harabasz,tamanios_clusters
48,agg-3-single-manhattan-spars-5,0.5,3,4,single,manhattan,"[chomebanking_transacciones, Visa_mpagominimo,...",40,0.959474,0.025342,905.205230,"{0: 11001, 1: 1, 2: 1, 3: 1}"
95,agg-4-average-manhattan-spars-8,0.8,4,5,average,manhattan,"[mautoservicio, ctarjeta_debito_transacciones,...",23,0.954495,0.122347,1392.857342,"{0: 10997, 1: 4, 2: 1, 3: 1, 4: 1}"
98,agg-4-single-manhattan-spars-8,0.8,4,5,single,manhattan,"[mautoservicio, ctarjeta_debito_transacciones,...",23,0.954495,0.122347,1392.857342,"{0: 10997, 1: 1, 2: 4, 3: 1, 4: 1}"
97,agg-4-single-euclidean-spars-8,0.8,4,5,single,euclidean,"[mautoservicio, ctarjeta_debito_transacciones,...",23,0.954495,0.122347,1392.857342,"{0: 10997, 1: 1, 2: 4, 3: 1, 4: 1}"
94,agg-4-average-euclidean-spars-8,0.8,4,5,average,euclidean,"[mautoservicio, ctarjeta_debito_transacciones,...",23,0.954495,0.122347,1392.857342,"{0: 10997, 1: 4, 2: 1, 3: 1, 4: 1}"
...,...,...,...,...,...,...,...,...,...,...,...,...
43,agg-3-complete-cosine-spars-5,0.5,3,4,complete,cosine,"[chomebanking_transacciones, Visa_mpagominimo,...",40,0.156616,2.100115,662.922540,"{0: 5252, 1: 204, 2: 3467, 3: 2081}"
6,agg-3-average-cosine-spars-2,0.2,3,4,average,cosine,"[Master_cadelantosefectivo, active_quarter, Vi...",43,0.068986,2.320653,629.215902,"{0: 3659, 1: 4313, 2: 846, 3: 2186}"
23,agg-5-complete-cosine-spars-2,0.2,5,6,complete,cosine,"[Master_cadelantosefectivo, active_quarter, Vi...",43,0.064252,2.152001,521.122540,"{0: 4800, 1: 2643, 2: 239, 3: 626, 4: 807, 5: ..."
13,agg-4-complete-cosine-spars-2,0.2,4,5,complete,cosine,"[Master_cadelantosefectivo, active_quarter, Vi...",43,-0.009368,2.180413,532.815143,"{0: 2925, 1: 2635, 2: 1999, 3: 2687, 4: 758}"


In [ ]:
pd.crosstab(
        df_baja_h["agg-5-complete-cosine-spars-5"],
        df_baja_h["ternaria"],
        margins=True
    )